In [1]:
# Colab setup: clone repo, install deps, cd into notebooks/ so "../data" paths work
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/khalilullahalfaath/credit-scoring-fairness-audit.git
    %cd credit-scoring-fairness-audit/notebooks
    !pip install -q -r ../requirements.txt

/content/credit-scoring-fairness-audit/notebooks


In [2]:
!which python

/usr/local/bin/python


In [3]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p ../data/raw
!cp "/content/drive/MyDrive/Projects/fairness-credit-scoring/"*.csv ../data/raw/

Mounted at /content/drive


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import fairlearn
import seaborn as sns

# Understanding dataset

from Kaggle Repo,

Problem Statement

You are working as a data scientist in a global finance company. Over the years, the company has collected basic bank details and gathered a lot of credit-related information. The management wants to build an intelligent system to segregate the people into credit score brackets to reduce the manual efforts.

Task

Given a person’s credit-related information, build a machine learning model that can classify the credit score.

## Understanding Data Structure

In this step, we want to know how big the data is and how it looks like

From Kaggle Repo, we get this data dictionary

| No | Kolom | Deskripsi |
|----|-------|-----------|
| 1 | `ID` | Represents a unique identification of an entry |
| 2 | `Customer_ID` | Represents a unique identification of a person |
| 3 | `Month` | Represents the month of the year |
| 4 | `Name` | Represents the name of a person |
| 5 | `Age` | Represents the age of the person |
| 6 | `SSN` | Represents the social security number of a person |
| 7 | `Occupation` | Represents the occupation of the person |
| 8 | `Annual_Income` | Represents the annual income of the person |
| 9 | `Monthly_Inhand_Salary` | Represents the monthly base salary of a person |
| 10 | `Num_Bank_Accounts` | Represents the number of bank accounts a person holds |
| 11 | `Num_Credit_Card` | Represents the number of other credit cards held by a person |
| 12 | `Interest_Rate` | Represents the interest rate on credit card |
| 13 | `Num_of_Loan` | Represents the number of loans taken from the bank |
| 14 | `Type_of_Loan` | Represents the types of loan taken by a person |
| 15 | `Delay_from_due_date` | Represents the average number of days delayed from the payment date |
| 16 | `Num_of_Delayed_Payment` | Represents the average number of payments delayed by a person |
| 17 | `Changed_Credit_Limit` | Represents the percentage change in credit card limit |
| 18 | `Num_Credit_Inquiries` | Represents the number of credit card inquiries |
| 19 | `Credit_Mix` | Represents the classification of the mix of credits |
| 20 | `Outstanding_Debt` | Represents the remaining debt to be paid (in USD) |
| 21 | `Credit_Utilization_Ratio` | Represents the utilization ratio of credit card |
| 22 | `Credit_History_Age` | Represents the age of credit history of the person |
| 23 | `Payment_of_Min_Amount` | Represents whether only the minimum amount was paid by the person |
| 24 | `Total_EMI_per_month` | Represents the monthly EMI payments (in USD) |
| 25 | `Amount_invested_monthly` | Represents the monthly amount invested by the customer (in USD) |
| 26 | `Payment_Behaviour` | Represents the payment behavior of the customer (in USD) |
| 27 | `Monthly_Balance` | Represents the monthly balance amount of the customer (in USD) |

In [12]:
df = pd.read_csv("../data/raw/train.csv")
df.head(5)

/tmp/ipykernel_4994/1081143885.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/train.csv")


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [13]:
df.tail(5)

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
99995,0x25fe9,CUS_0x942c,April,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,_,502.38,34.663572,31 Years and 6 Months,No,35.104023,60.97133255718485,High_spent_Large_value_payments,479.866228,Poor
99996,0x25fea,CUS_0x942c,May,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,_,502.38,40.565631,31 Years and 7 Months,No,35.104023,54.18595028760385,High_spent_Medium_value_payments,496.65161,Poor
99997,0x25feb,CUS_0x942c,June,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,Good,502.38,41.255522,31 Years and 8 Months,No,35.104023,24.02847744864441,High_spent_Large_value_payments,516.809083,Poor
99998,0x25fec,CUS_0x942c,July,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,Good,502.38,33.638208,31 Years and 9 Months,No,35.104023,251.67258219721603,Low_spent_Large_value_payments,319.164979,Standard
99999,0x25fed,CUS_0x942c,August,Nicks,25,078-73-5990,Mechanic,39628.99_,3359.415833,4,...,Good,502.38,34.192463,31 Years and 10 Months,No,35.104023,167.1638651610451,!@9#%8,393.673696,Poor


In [14]:
df.shape

(100000, 28)

In [15]:
df.columns

Index(['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation',
       'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
       'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan',
       'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt',
       'Credit_Utilization_Ratio', 'Credit_History_Age',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score'],
      dtype='object')

Berdasarkan quick view, dataset ini terdiri dari data-data customer bank dalam satu bulan. Data terdiri 100.000 baris dan 28 kolom.

## Understanding Data type

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

There are 28 columns with 8 columns have null values. Most features are str (object). Ada data numerik, tetapi dianggap sebagai object

## Understanding numerical summary

In [17]:
df.describe()

,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_Credit_Inquiries,Credit_Utilization_Ratio,Total_EMI_per_month
count,84998.000000,100000.000000,100000.00000,100000.000000,100000.000000,98035.000000,100000.000000,100000.000000
mean,4194.170850,17.091280,22.47443,72.466040,21.068780,27.754251,32.285173,1403.118217
std,3183.686167,117.404834,129.05741,466.422621,14.860104,193.177339,5.116875,8306.041270
min,303.645417,-1.000000,0.00000,1.000000,-5.000000,0.000000,20.000000,0.000000
25%,1625.568229,3.000000,4.00000,8.000000,10.000000,3.000000,28.052567,30.306660
50%,3093.745000,6.000000,5.00000,13.000000,18.000000,6.000000,32.305784,69.249473
75%,5957.448333,7.000000,7.00000,20.000000,28.000000,9.000000,36.496663,161.224249
max,15204.633333,1798.000000,1499.00000,5797.000000,67.000000,2597.000000,50.000000,82331.000000


For numerical data, Monthly_Inhand_Salary have null values.

## Understanding cardinality for categorical data

In [18]:
obj = df.select_dtypes('object')
obj.nunique().sort_values(ascending=False)

,0
ID,100000
Monthly_Balance,98792
Amount_invested_monthly,91049
Annual_Income,18940
Outstanding_Debt,13178
SSN,12501
Customer_ID,12500
Name,10139
Type_of_Loan,6260
Changed_Credit_Limit,4384


Based on the table above, this dataset has two unique IDs: (i) `ID`, which is unique for each row and represents one customer record (per month), and (ii) `Customer_ID`, which represents the ID of one customer. There are 8 unique months. Next, we need to confirm whether the grain "customer x month" holds, based on our assumption that 12,500 x 8 = 100,000 (row count).

From kaggle, we know that our target is `Credit_Score`.

List of identifier:
1. `ID`
2. `Customer_ID`
3. `Name`
4. `SSN`

Based on the data dictionary, except for four columns mentioned above (`ID`, `Customer_ID`, `Month`, and `Credit_Score`) and identified identifier columns, the following columns are categorically valid:
1.  `Payment_of_Min_Amount`
2. `Occupation`
3. `Payment_Behaviour`
4. `Credit_Mix`
5. `Month`
6. `Credit_Score` (Target Column)

On the other hand, there are several problems we have to handle next:
1. Wrong datatypes: `Monthly_Balance`, `Amount_Invested_Monthly`, `Age`, `Annual_Income`, `Outstanding_Debt`, `Num_of_Loan`, `Num_of_Delayed_Payment`, `Changed_Credit_Limit`, and `Credit_History_Age`.
2. In addition, `Age` has 1,788 unique values, far more than the plausible range of adult ages.
3. `SSN` has 12,501 unique values, which is one more than the number of customers.
4. The `Annual_Income` column has 18,940 unique values, which is more than the number of customers.
5. The `Type_of_Loan` column contains multiple values in a single cell (6,260 combinations).




In [31]:
categorically_valid = ["Payment_of_Min_Amount", "Occupation", "Payment_Behaviour", "Credit_Mix", "Month", "Credit_Score"]

for cat in categorically_valid:
    counts = df[cat].value_counts(dropna=False)
    pct = df[cat].value_counts(dropna=False, normalize=True).mul(100).round(1)
    print(pd.DataFrame({"count": counts, "percent (%)": pct}), "\n")

                       count  percent (%)
Payment_of_Min_Amount                    
Yes                    52326         52.3
No                     35667         35.7
NM                     12007         12.0 

               count  percent (%)
Occupation                       
_______         7062          7.1
Lawyer          6575          6.6
Architect       6355          6.4
Engineer        6350          6.4
Scientist       6299          6.3
Mechanic        6291          6.3
Accountant      6271          6.3
Developer       6235          6.2
Media_Manager   6232          6.2
Teacher         6215          6.2
Entrepreneur    6174          6.2
Doctor          6087          6.1
Journalist      6085          6.1
Manager         5973          6.0
Musician        5911          5.9
Writer          5885          5.9 

                                  count  percent (%)
Payment_Behaviour                                   
Low_spent_Small_value_payments    25513         25.5
High_spent_Medi

Based on the data above, we know that there is a unique placeholder value in several columns, i.e `Occupation`, using `_______`; `Payment_Behaviour`, using `!@9#%8`; and `Credit_Mix`, using `_`. These placeholder values, won't be captured by isna(), so we have to change it to NaN.

| Column            | Placeholder | Count  | Percentage |
|-------------------|-------------|--------|------------|
| `Occupation`      | `_______`   | 7,062  | 7.1%       |
| `Payment_Behaviour` | `!@9#%8`  | 7,600  | 7.6%       |
| `Credit_Mix`      | `_`         | 20,195 | 20.2%      |

In addition, `Payment_Of_Min_Amount` have NM value, possibly short for `Not mentioned`, to be decided. Also, `Credit_Score` as our target value is not balanced.

Finally, for `Month` column, we want to verify whether `Customer_ID` x `Month` is indeed our grain.

### Checking grain
In this step, we want to confirm whether `Customer_ID` x `Month` is indeed our grain.

In [38]:
# checking grain
print("Number of duplicate Customer_ID x Month\t:", df.duplicated(['Customer_ID', 'Month']).sum())
print("Rows per Customer_ID\t\t:", df.groupby("Customer_ID").size().value_counts())


Number of duplicate Customer_ID x Month	: 0
Rows per Customer_ID		: 8    12500
Name: count, dtype: int64


Grain is confirmed: there are no duplicate `Customer_ID` x `Month` pairs, and each of the 12,500 customers has exactly 8 rows, one for each month.

Grain: one row per customer per month

### Checking NM in `Payment_of_Min_Amount`




In [39]:
pd.crosstab(df.Payment_of_Min_Amount, df.Credit_Score, normalize='index').round(3)

Credit_Score,Good,Poor,Standard
Payment_of_Min_Amount,,,
NM,0.187,0.293,0.520
No,0.383,0.134,0.483
Yes,0.037,0.396,0.568


In [40]:
pd.crosstab(df.Payment_of_Min_Amount, df.Num_of_Loan.astype(str).str.strip('_').eq('0'))

Num_of_Loan,False,True
Payment_of_Min_Amount,,
NM,10764,1243
No,28668,6999
Yes,49638,2688


`NM` is retained as its own category. Its `Credit_Score` distribution (`Good` 18.7%, `Poor` 29.3%, `Standard` 52.0%) is close to the overall distribution and unlike `Yes` or `No`, so imputing it would invent a payment behaviour we cannot verify. Its meaning is not defined in the data dictionary.

### Checking whether `Occupation` and `Credit_Mix` are same for all Customer_ID history

In this step, we want to check whether `Occupation` and `Credit_Mix` have the same values for each grain

In [43]:
# check whether Occupation and Credit_Mix are always have the same value for all customer history
col, ph = 'Occupation', '_______'
cnt = df[col].eq(ph).groupby(df['Customer_ID']).sum()   # jumlah placeholder per pelanggan (0..8)
cnt.value_counts().sort_index()

,count
Occupation,
0,6950
1,4248
2,1113
3,169
4,19
5,1


In [44]:
col, ph = 'Credit_Mix', '_'
cnt = df[col].eq(ph).groupby(df['Customer_ID']).sum()   # jumlah placeholder per pelanggan (0..8)
cnt.value_counts().sort_index()

,count
Credit_Mix,
0,2023
1,4236
2,3625
3,1902
4,583
5,115
6,16


In [45]:
real = df[df['Occupation'] != '_______']
real.groupby('Customer_ID')['Occupation'].nunique().value_counts()

,count
Occupation,
1,12500


In [46]:
real = df[df['Credit_Mix'] != '_']
real.groupby('Customer_ID')['Credit_Mix'].nunique().value_counts()

,count
Credit_Mix,
1,12500


After counting placeholders per customer (out of 8 months), shows that they are spread across rows rather than concentrated in specific customers. No customer has all 8 months affected: the maximum is 5 for Occupation and 6 for Credit_Mix. The per-customer distribution is close to what independent row-level corruption would produce (about 6,958 and 4,228 customers expected with 0 and 1 placeholders in Occupation, versus 6,950 and 4,248 observed), which is consistent with placeholders appearing at random, though no formal test was run.

Stability. Among non-placeholder rows, every one of the 12,500 customers has exactly one distinct value in Occupation and exactly one in Credit_Mix. Both attributes are therefore constant per customer in this dataset.

Implication. Because each customer has at least one valid value and the attribute does not change across months, every placeholder can be recovered by copying the customer's own valid value. This is a deterministic recovery, not imputation: no value is estimated, so no information from other customers is introduced.

### Checking whether `Credit_Score` changes for one customer

Next, we check whether `Credit_Score` value is always unique for one customer

In [ ]:
df.groupby